# Tracks and Multi-Layer Composition
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/editor/feature/tracks_layering_model.ipynb)

---

Video editing is fundamentally about layering. A professional video isn't just one clip playing—it's multiple layers stacked together: background footage, music, graphics, text, and captions all composing into a single output.

In VideoDB Editor, **Tracks** are the mechanism for this layering. Think of Tracks like layers in Photoshop or After Effects. Each Track is a separate layer that can hold multiple clips, and these tracks stack vertically to create complex compositions.

Understanding tracks is essential because they control both the **horizontal timeline** (when things play) and the **vertical z-order** (what appears on top of what). Master tracks, and you can build anything from simple videos with subtitles to complex multi-camera productions with graphics, titles, and effects.

**What you'll learn:**

- How Tracks work as layers in the Editor's composition model
- The difference between vertical stacking (z-order/layers) and horizontal stacking (timeline sequencing)
- The critical "double start" concept: trimming vs timing
- How to build multi-layer compositions step-by-step (video + audio + overlays + text + captions)
- How track order affects final rendering (z-order rules)
- Practical patterns for layering different asset types (Video, Audio, Image, Text, Caption)
- Real-world composition examples you can adapt for your projects

By the end of this notebook, you'll understand how to orchestrate multiple media elements across tracks to create professional, layered video compositions.

---

## 📦 Step 1: Install VideoDB Editor SDK

Lets install VideoDB SDK

In [1]:
!pip -q install videodb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.7 MB/s eta 0:00:00


---

## 🔌 Step 2: Connect to VideoDB

We'll establish a connection to VideoDB and access our collection.

In [2]:
import videodb
import os
from videodb import play_stream
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
print("✓ Connected to VideoDB")

coll = conn.get_collection()
print("✓ Collection ready")

Please enter your VideoDB API Key: ··········
✓ Connected to VideoDB
✓ Collection ready


---

## 📥 Step 3: Upload Assets

For a complete multi-layer composition, we need different types of assets:
- **Video**: The main footage (base layer)
- **Audio**: Background music
- **Image**: A logo or watermark overlay

We'll upload these assets now. This may take a few moments.

In [3]:
# Upload main video
print("Uploading main video...")
video = coll.upload(url="https://www.youtube.com/watch?v=LPZh9BOjkQs")
print(f"✓ Video uploaded: {video.id}")

Uploading main video...
✓ Video uploaded: m-z-019edf09-00f1-7453-8a7a-64eb99011165


In [4]:
from videodb import MediaType

# Upload background audio
print("Uploading background audio...")
audio = coll.upload(
    url="https://www.youtube.com/watch?v=Q7HjxOAU5Kc",
    media_type=MediaType.audio
)
print(f"Audio uploaded: {audio.id}")


Uploading background audio...
Audio uploaded: a-z-019edf09-ce7c-72a2-b78a-4d2de0a41603


In [5]:
# Upload image for overlay
print("Uploading image overlay...")
image = coll.upload(url="https://picsum.photos/400/400")
print(f"✓ Image uploaded: {image.id}")

Uploading image overlay...
✓ Image uploaded: img-z-019edf09-e612-7cc0-9164-a81ce74827a6


**Tip**: If you need to re-run this notebook, you can skip the upload cells and use these instead:

```python
# video = coll.get_video("your_video_id")
# audio = coll.get_audio("your_audio_id")
# image = coll.get_image("your_image_id")
```

---

## 🎬 Step 4: Import Editor Components

Now we'll import the core Editor objects:
- **Timeline**: The canvas (defines resolution and background)
- **Track**: A layer that holds clips
- **Clip**: A wrapper that positions an asset in time and space
- **Assets**: VideoAsset, AudioAsset, ImageAsset, TextAsset, CaptionAsset

In [6]:
from videodb.editor import (
    Timeline, Track, Clip,
    VideoAsset, AudioAsset, ImageAsset, TextAsset, CaptionAsset,
    Position, Fit,
    Font, Alignment, HorizontalAlignment, VerticalAlignment,
    CaptionAnimation, FontStyling, Positioning, CaptionAlignment
)

print("✓ Editor components imported")

✓ Editor components imported


---

## 🎯 Understanding the Track Layering Model

Before we start building, let's understand the key concepts:

### Vertical Stacking (Z-Order)
- Tracks are added to the Timeline in sequence
- **Later tracks render on top of earlier tracks**
- This creates a layering effect (like Photoshop layers)

### Horizontal Stacking (Timeline Position)
- Within a Track, clips can be placed at different times: `track.add_clip(start=X, clip)`
- The `start` parameter controls **when** the clip appears on the timeline

### The "Double Start" Concept
There are TWO different `start` parameters:
1. **Asset-level start** (TRIMMING): `VideoAsset(start=10)` skips the first 10 seconds of the source file
2. **Track-level start** (TIMING): `track.add_clip(start=5, clip)` places the clip at the 5-second mark of the timeline

Now let's see this in action!

---

## 🎥 Layer 1: Base Video Track

We'll start with a single track containing our main video.

This establishes the foundation — a simple timeline with one video playing for 15 seconds.

In [7]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Create a video clip (muted so we can add our own audio later)
video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        start=0,
        volume=0  # Mute the original video audio
    ),
    duration=15
)

# Add the clip to a track
video_track = Track()
video_track.add_clip(0, video_clip)  # Start at 0 seconds

# Add the track to the timeline
timeline.add_track(video_track)

print("✓ Layer 1 added: Base video (muted)")

✓ Layer 1 added: Base video (muted)


Let's generate and preview this single-layer composition:

In [8]:
stream_url_layer1 = timeline.generate_stream()
print(f"Stream URL (1 layer - video only): {stream_url_layer1}")
play_stream(stream_url_layer1)

Stream URL (1 layer - video only): https://play.videodb.io/v1/690008c9-4d6f-40ff-87d5-feec8f06c8e9.m3u8


**What we see**: A simple 15-second video with no audio. This is our base layer.

---

## 🎵 Layer 2: Background Audio Track

Now we'll add a second track with background music.

Since the video is muted, this audio will be the only sound in our composition.

**Important**: Audio has no visual component, so it won't obscure the video below it.

In [9]:
# Create an audio clip
audio_clip = Clip(
    asset=AudioAsset(
        id=audio.id,
        volume=0.3  # Lower volume so it doesn't overpower
    ),
    duration=15
)

# Add to a new track
audio_track = Track()
audio_track.add_clip(0, audio_clip)

# Add this track to the timeline
timeline.add_track(audio_track)

print("✓ Layer 2 added: Background audio")

✓ Layer 2 added: Background audio


Let's generate the stream with 2 layers (video + audio):

In [10]:
stream_url_layer2 = timeline.generate_stream()
print(f"Stream URL (2 layers - video + audio): {stream_url_layer2}")
play_stream(stream_url_layer2)

Stream URL (2 layers - video + audio): https://play.videodb.io/v1/6aecc27b-abbe-4e3c-aafa-722604173b7a.m3u8


**What we see**: The same video, but now with background music playing. The audio track sits "above" the video track in the layer stack, but since audio is invisible, we only hear it.

---

## 🖼️ Layer 3: Image Overlay Track

Now we'll add a third track with an image overlay (like a logo or watermark).

We'll position it in the **top-right corner** and scale it down so it doesn't cover too much of the video.

**Z-Order in action**: This track is added after the video track, so the image will render **on top** of the video.

In [11]:
# Create an image clip (small logo in corner)
image_clip = Clip(
    asset=ImageAsset(id=image.id),
    duration=15,
    position=Position.top_right,
    fit=Fit.none,
    scale=0.15,  # Scale down to 15% of original size
    opacity=0.8  # Slightly transparent
)

# Add to a new track
image_track = Track()
image_track.add_clip(0, image_clip)

# Add this track to the timeline
timeline.add_track(image_track)

print("✓ Layer 3 added: Image overlay (top-right corner)")

✓ Layer 3 added: Image overlay (top-right corner)


Let's generate the stream with 3 layers (video + audio + image):

In [12]:
stream_url_layer3 = timeline.generate_stream()
print(f"Stream URL (3 layers - video + audio + image): {stream_url_layer3}")
play_stream(stream_url_layer3)

Stream URL (3 layers - video + audio + image): https://play.videodb.io/v1/4e6d0733-bc87-488d-b2de-892ac0d254cc.m3u8


**What we see**: The video with background music, and now a small logo/watermark in the top-right corner. The image layer renders on top of the video layer because it was added later.

---

## 📝 Layer 4: Text Overlay Track

Now we'll add a fourth track with a text title.

We'll position it at the **top-center** with a semi-transparent background box for readability.

**Key point**: This text track is added after the image track, so if they overlap, the text will appear on top.

In [13]:
from videodb.editor import Background, TextAlignment, Border

# Create a text clip
text_asset = TextAsset(
    text="Track Layering Demo",
    font=Font(
        family="Clear Sans",
        size=48,
        color="#FFFFFF"
    ),
    border=Border(color="#000000", width=2.0),
    background=Background(
        width=600,
        height=80,
        color="#1a1a1a",
        opacity=0.7,
        text_alignment=TextAlignment.center
    ),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

text_clip = Clip(
    asset=text_asset,
    duration=15
)

# Add to a new track
text_track = Track()
text_track.add_clip(0, text_clip)

# Add this track to the timeline
timeline.add_track(text_track)

print("✓ Layer 4 added: Text overlay (top-center)")

✓ Layer 4 added: Text overlay (top-center)


Let's generate the stream with 4 layers (video + audio + image + text):

In [14]:
stream_url_layer4 = timeline.generate_stream()
print(f"Stream URL (4 layers - video + audio + image + text): {stream_url_layer4}")
play_stream(stream_url_layer4)

Stream URL (4 layers - video + audio + image + text): https://play.videodb.io/v1/ef470cf8-f5f2-40bb-8eb5-d5c19fef96f0.m3u8


**What we see**: The video with background music, logo in the corner, and now a title at the top. We're building up layers!

---

## 💬 Layer 5: Caption Track

Finally, we'll add a fifth track with auto-generated captions.

By setting `src="auto"`, VideoDB will automatically transcribe the speech in the video and create time-synced subtitles.

We'll use the `supersize` animation where the active word grows larger as it's spoken.

**Note**: Since the original video is muted in our composition, captions will be generated from the original video's audio (before muting).

In [15]:
# CaptionAsset with src='auto' requires spoken word index
print("Indexing spoken words for auto-captions...")
video.index_spoken_words()
print("Spoken word index complete!")


Indexing spoken words for auto-captions...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:15<00:00,  6.25it/s]

Spoken word index complete!


In [16]:
# Create a caption clip with auto-generated subtitles
caption_asset = CaptionAsset(
    src="auto",  # Auto-generate from video audio
    animation=CaptionAnimation.supersize,
    primary_color="&H00FFFFFF",  # White text (ASS format)
    secondary_color="&H0000D7FF",  # Orange for active word
    position=Positioning(
        alignment=CaptionAlignment.bottom_center
    ),
    font=FontStyling(
        bold=True,
        size=32
    )
)

caption_clip = Clip(
    asset=caption_asset,
    duration=15
)

# Add to a new track
caption_track = Track()
caption_track.add_clip(0, caption_clip)

# Add this track to the timeline (topmost layer)
timeline.add_track(caption_track)

print("✓ Layer 5 added: Auto-generated captions (bottom-center)")

✓ Layer 5 added: Auto-generated captions (bottom-center)


/tmp/ipykernel_1318/1775089903.py:2: UserWarning: CaptionAsset(src='auto'): the video must be indexed (e.g. video.index_spoken_words()) for captions to be generated.
  caption_asset = CaptionAsset(


Let's generate the final stream with all 5 layers:

In [17]:
stream_url_final = timeline.generate_stream()
print(f"Stream URL (5 layers - complete composition): {stream_url_final}")
play_stream(stream_url_final)

Stream URL (5 layers - complete composition): https://play.videodb.io/v1/aca3c49c-9522-43a6-8fec-c374fc20773e.m3u8


**What we see**: A complete multi-layer composition!
- Base video (Layer 1)
- Background music (Layer 2)
- Logo watermark in corner (Layer 3)
- Title text at top (Layer 4)
- Animated captions at bottom (Layer 5)

Each layer was added in sequence, creating a professional-looking video with multiple elements.

---

## 🔄 Demonstrating the "Double Start" Concept

Let's clarify the difference between **trimming** and **timing** using a practical example.

We'll create a new timeline that shows:
- A video clip that starts at the 5-second mark of the source (TRIMMING)
- But appears at the 3-second mark of the timeline (TIMING)

In [18]:
timeline_double_start = Timeline(conn)
timeline_double_start.background = "#2B2B2B"

# Asset-level start (TRIMMING): Skip first 5 seconds of source video
trimmed_video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        start=5,  # Start playback at 5-second mark of source
        volume=0.5
    ),
    duration=10  # Play for 10 seconds (from 5s to 15s of source)
)

double_start_track = Track()
# Track-level start (TIMING): Place clip at 3-second mark of timeline
double_start_track.add_clip(3, trimmed_video_clip)

timeline_double_start.add_track(double_start_track)

print("✓ Demo created: Video plays seconds 5-15 of source, starting at 3s on timeline")

✓ Demo created: Video plays seconds 5-15 of source, starting at 3s on timeline


In [19]:
stream_url_double_start = timeline_double_start.generate_stream()
print(f"Stream URL (double start demo): {stream_url_double_start}")
play_stream(stream_url_double_start)

Stream URL (double start demo): https://play.videodb.io/v1/1bbcc579-30a2-4f43-8858-4312eb0574a9.m3u8


**What we see**:
- The timeline starts with 3 seconds of blank background
- Then the video appears, but it's showing the content from the 5-second mark of the source file
- This demonstrates the two independent `start` parameters working together

---

## 🎯 Z-Order: Track Order Matters

Let's demonstrate how track order affects the final rendering.

We'll create two timelines with the same content but in different track orders.

### Example A: Text added BEFORE image

In [20]:
timeline_a = Timeline(conn)
timeline_a.background = "#2B2B2B"

# Base video
base_video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=10
)
base_track = Track()
base_track.add_clip(0, base_video_clip)
timeline_a.add_track(base_track)

# Add TEXT first
text_clip_a = Clip(
    asset=TextAsset(
        text="Text Layer",
        font=Font(family="Clear Sans", size=60, color="#FF0000"),
        alignment=Alignment(
            horizontal=HorizontalAlignment.center,
            vertical=VerticalAlignment.center
        )
    ),
    duration=10
)
text_track_a = Track()
text_track_a.add_clip(0, text_clip_a)
timeline_a.add_track(text_track_a)

# Add IMAGE second (will render on top)
image_clip_a = Clip(
    asset=ImageAsset(id=image.id),
    duration=10,
    position=Position.center,
    fit=Fit.none,
    scale=0.3,
    opacity=0.9
)
image_track_a = Track()
image_track_a.add_clip(0, image_clip_a)
timeline_a.add_track(image_track_a)

print("✓ Timeline A: Text added first, then image (image will be on top)")

✓ Timeline A: Text added first, then image (image will be on top)


In [21]:
stream_url_a = timeline_a.generate_stream()
print(f"Stream URL (Timeline A - image on top): {stream_url_a}")
play_stream(stream_url_a)

Stream URL (Timeline A - image on top): https://play.videodb.io/v1/f40193fa-0eeb-4a7c-baab-beffe800bfc8.m3u8


### Example B: Image added BEFORE text

In [22]:
timeline_b = Timeline(conn)
timeline_b.background = "#2B2B2B"

# Base video
base_video_clip_b = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=10
)
base_track_b = Track()
base_track_b.add_clip(0, base_video_clip_b)
timeline_b.add_track(base_track_b)

# Add IMAGE first
image_clip_b = Clip(
    asset=ImageAsset(id=image.id),
    duration=10,
    position=Position.center,
    fit=Fit.none,
    scale=0.3,
    opacity=0.9
)
image_track_b = Track()
image_track_b.add_clip(0, image_clip_b)
timeline_b.add_track(image_track_b)

# Add TEXT second (will render on top)
text_clip_b = Clip(
    asset=TextAsset(
        text="Text Layer",
        font=Font(family="Clear Sans", size=60, color="#FF0000"),
        alignment=Alignment(
            horizontal=HorizontalAlignment.center,
            vertical=VerticalAlignment.center
        )
    ),
    duration=10
)
text_track_b = Track()
text_track_b.add_clip(0, text_clip_b)
timeline_b.add_track(text_track_b)

print("✓ Timeline B: Image added first, then text (text will be on top)")

✓ Timeline B: Image added first, then text (text will be on top)


In [23]:
stream_url_b = timeline_b.generate_stream()
print(f"Stream URL (Timeline B - text on top): {stream_url_b}")
play_stream(stream_url_b)

Stream URL (Timeline B - text on top): https://play.videodb.io/v1/4cfc102c-2e9f-4237-b484-a8c1c5152e16.m3u8


**What we see**:
- In Timeline A, the image obscures the text (image was added last)
- In Timeline B, the text is clearly visible on top of the image (text was added last)

This demonstrates the fundamental rule: **tracks added later render on top of tracks added earlier**.

---

## 🎓 Recap: What We Learned

In this notebook, we explored how Tracks work in VideoDB Editor:

### Key Concepts

1. **Tracks as Layers**: Each Track is a separate layer that can contain multiple clips

2. **Vertical Stacking (Z-Order)**:
   - Tracks added later render on top of earlier tracks
   - This creates layered compositions (video + overlays + text + captions)

3. **Horizontal Stacking (Timeline Position)**:
   - Within a track, clips can be placed at different times using `track.add_clip(start=X, clip)`
   - This controls when clips appear in the final video

4. **The "Double Start" Concept**:
   - `VideoAsset(start=10)` = TRIMMING (skip first 10s of source)
   - `track.add_clip(start=5, clip)` = TIMING (place clip at 5s on timeline)
   - These are independent and work together

5. **Multi-Layer Compositions**:
   - We built a 5-layer composition: video + audio + image + text + captions
   - Each layer serves a specific purpose in the final output
   - Professional videos often use 3-7 tracks

### Practical Applications

- **Social media videos**: Background footage + music + logo + captions
- **Tutorials**: Screen recording + webcam overlay + title cards + annotations
- **Marketing videos**: Product footage + background music + branding + text CTAs

### Next Steps

Now that you understand tracks, try experimenting:
- Add more simultaneous clips on the same track (split-screen effects)
- Create picture-in-picture by scaling and positioning video clips
- Layer multiple text overlays at different times
- Combine tracks with transitions and filters for complex effects

The track system is the foundation of all complex video editing in VideoDB Editor!